# Limpieza básica de los GRD respiratorios (2019-2025)

Este notebook toma los `GRD_PUBLICO_AAAA_ER.txt` (ya filtrados por `DIAGNOSTICO1` que empieza con `J`) y deja un `.parquet` limpio por año en `GRD_CIE_TM/GRD_limpio/`.

**Qué hace, en orden**

1. Lectura con la codificación correcta de cada año y limpieza de texto (espacios, vacíos).
2. Unificación de nombres de columnas entre años (`CIP_ENCRIPTADO` / `ID_BENEFICIARIO`).
3. **Duplicados** (con justificación).
4. **Nulos y valores "falsos nulos"** (con justificación).
5. Fechas a un único tipo `datetime64` (se muestran como `dd/mm/yyyy`).
6. Variables del proyecto: `SEXO`, `DIAGNOSTICO1`, `PROCEDIMIENTO1-30`, `FECHA_INGRESO`, `MES_INGRESO`, `TRAMO_EDAD`, `HOSPITAL`, `COMUNA`.
7. Reporte de calidad (antes/después) y exportación a parquet.

> **Ojo con el año:** cada archivo `GRD_PUBLICO_AAAA` contiene los egresos con **fecha de alta** en ese año; por eso ~1% de los `FECHA_INGRESO` caen en el año anterior (el 2025 llega hasta 2023). Para análisis temporal por mes de ingreso, hay que concatenar todos los años y agrupar por `FECHA_INGRESO`, no por `ANIO`.

> Las variables auxiliares `EDAD` y `FECHA_NACIMIENTO` se usan solo para calcular `TRAMO_EDAD` y se eliminan antes de formar el GRD final.

In [15]:
import re
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------- Configuración ----------------
DIR_GRD     = Path('../GRD_CIE_TM')

ANIOS = range(2019, 2025)

# Codificación real de cada archivo _ER.txt (verificada: 2019-2024 NO son utf-8 válido; 2025 sí lo es).
ENCODING = {anio: 'latin-1' for anio in ANIOS}

# Si True, FECHA_INGRESO se guarda como texto 'dd/mm/yyyy'.
# Recomendado False: en parquet se guardan como fecha real (se pueden filtrar y ordenar) y
# el formato dd/mm/yyyy se aplica solo al mostrarlas (ver última celda).
FECHAS_COMO_TEXTO = False

## 1. Catálogos (CIE-10, CIE-9, tabla maestra de hospitales)

Dos detalles encontrados al revisar los datos:

* **CIE-9**: el catálogo trae el código con dos dígitos enteros (`00.17`, `03.31`), pero el GRD trae `0.17`, `3.31`. Sin corregir esto, esos procedimientos no calzarían. Se normaliza con un cero a la izquierda.
* **Hospitales**: la tabla maestra solo trae *código* y *nombre*, no la comuna.

In [16]:
def limpiar_texto(s):
    """strip + vacíos -> NA"""
    s = s.astype('string').str.strip()
    return s.mask(s == '')

# --- CIE-10 (diagnósticos) ---
cie10 = pd.read_excel(DIR_GRD / 'CIE-10.xlsx', dtype=str)
cie10['Código'] = limpiar_texto(cie10['Código']).str.upper()
DESC_CIE10 = cie10.drop_duplicates('Código').set_index('Código')['Descripción']

# --- Tabla maestra de hospitales ---
maestra = pd.read_excel(DIR_GRD / 'Tablas maestras bases GRD .xlsx', sheet_name='Hospitales')
maestra.columns = ['COD_HOSPITAL', 'HOSPITAL']
maestra['COD_HOSPITAL'] = limpiar_texto(maestra['COD_HOSPITAL'])
maestra['HOSPITAL'] = limpiar_texto(maestra['HOSPITAL'])
NOMBRE_HOSPITAL = maestra.drop_duplicates('COD_HOSPITAL').set_index('COD_HOSPITAL')['HOSPITAL']

# --- Región por comuna (catálogo versionado en el repositorio) ---
RUTA_CATALOGO = Path('catalogo_comuna_region.csv')
catalogo = pd.read_csv(RUTA_CATALOGO, dtype=str, encoding='utf-8-sig')
catalogo['COMUNA'] = limpiar_texto(catalogo['COMUNA']).str.upper()
assert catalogo['COMUNA'].is_unique, 'Hay comunas repetidas en el catálogo'
REGION_POR_COMUNA = catalogo.set_index('COMUNA')['REGION'].to_dict()

## 3. Funciones de limpieza

### Decisiones y justificación

**Duplicados**
* Se **eliminan solo los duplicados exactos** (las 129 columnas idénticas). Un registro idéntico en todo, incluyendo fechas de traslado, alta, GRD y médico encriptado, es un error de extracción, no dos hospitalizaciones reales. Esto afecta casi solo a 2020 (~128 filas) y 2022 (2 filas).
* Los registros que coinciden en `ID + hospital + fecha de ingreso + DIAGNOSTICO1` pero difieren en otras columnas se consideran posibles duplicados y **se eliminan**. El total eliminado queda registrado en el reporte de calidad.

**Nulos**
* **Falsos nulos de texto**: `DESCONOCIDO`/`DESCONOCIDA`/`NO APLICA` dentro de columnas de fecha o de procedimiento se convierten en nulo real.
* **Nulos estructurales** (`PROCEDIMIENTO2-30`, `DIAGNOSTICO2-35`, `FECHATRASLADO*`, `*RN*`): el vacío significa "no aplica" (el paciente no tuvo ese procedimiento o traslado). **No se imputan ni se eliminan filas.** Incluye `PROCEDIMIENTO1` (casi el 100% de sus nulos son pacientes sin procedimientos).
* **Falsos nulos** (`DESCONOCIDO`, `NO APLICA` dentro de columnas de fecha): se convierten a nulo real (`NaT`), porque una fecha no puede ser "desconocido".
* **`SEXO = DESCONOCIDO`** (<0.02% de las filas): se conserva como categoría explícita. Es información, y es consistente con `TRAMO_EDAD = DESCONOCIDO` del GRD 2025.
* Las filas con `FECHA_INGRESO` nula se eliminan, porque `MES_INGRESO` depende de una fecha válida. Los nulos de identificación y fechas de nacimiento se informan sin imputarlos.

In [17]:
FALSOS_NULOS = {'DESCONOCIDO', 'DESCONOCIDA', 'NO APLICA', 'NO INFORMADO', 'S/I', 'SIN INFORMACION', 'SIN INFORMACIÓN'}

COLS_NUM_COMA = ['IR_29301_PESO', 'PESORN1', 'PESORN2', 'PESORN3', 'PESORN4']
COLS_NUM = ['RN1ESTADO', 'RN2ESTADO', 'RN3ESTADO', 'RN4ESTADO', 'USOSPABELLON',
            'IR_29301_SEVERIDAD', 'IR_29301_MORTALIDAD']

def parsear_fecha(serie):
    """Acepta 'yyyy-mm-dd' y 'dd-mm-yyyy' (formato por año/columna distinto). Lo demás -> NaT."""
    s = limpiar_texto(serie)
    s = s.mask(s.str.upper().isin(FALSOS_NULOS))
    es_iso = s.str.match(r'^\d{4}-\d{2}-\d{2}').fillna(False).astype(bool)
    es_dmy = s.str.match(r'^\d{2}-\d{2}-\d{4}').fillna(False).astype(bool)
    f_iso = pd.to_datetime(s.where(es_iso).str[:10], format='%Y-%m-%d', errors='coerce')
    f_dmy = pd.to_datetime(s.where(es_dmy).str[:10], format='%d-%m-%Y', errors='coerce')
    return f_iso.fillna(f_dmy)

def limpiar_grd(anio):
    ruta = DIR_GRD / f'GRD_PUBLICO_{anio}_ER.txt'
    df = pd.read_csv(ruta, sep='|', encoding=ENCODING[anio], dtype=str)
    rep = {'anio': anio, 'filas_originales': len(df)}

    # --- 1. Texto: strip y vacíos -> NA ---
    for c in df.columns:
        df[c] = limpiar_texto(df[c])

    # --- 2. Unificar nombre del ID (2019-2023: CIP_ENCRIPTADO; 2024-2025: ID_BENEFICIARIO) ---
    df = df.rename(columns={'CIP_ENCRIPTADO': 'ID_BENEFICIARIO'})

    # --- 3. Duplicados exactos ---
    n = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    rep['duplicados_exactos_eliminados'] = n - len(df)

    # --- 4. Fechas (todas las columnas FECHA*) -> datetime64 ---
    cols_fecha = [c for c in df.columns if c.startswith('FECHA')]
    for c in cols_fecha:
        crudo = df[c]
        df[c] = parsear_fecha(crudo).astype('datetime64[ns]')     # misma unidad en todos los años
        malas = crudo.notna() & df[c].isna() & ~crudo.str.upper().isin(FALSOS_NULOS)
        if malas.any():
            rep[f'fecha_no_parseable_{c}'] = int(malas.sum())
    rep['FECHA_INGRESO_nula'] = int(df['FECHA_INGRESO'].isna().sum())
    df = df.dropna(subset=['FECHA_INGRESO']).reset_index(drop=True)
    df['MES_INGRESO'] = df['FECHA_INGRESO'].dt.month.astype('Int64')
    rep['FECHA_INGRESO_fuera_del_anio'] = int((df['FECHA_INGRESO'].dt.year != anio).sum() - df['FECHA_INGRESO'].isna().sum())

    # --- 5. Numéricos (coma decimal, ceros a la izquierda del código GRD) ---
    for c in COLS_NUM_COMA:
        if c in df: df[c] = pd.to_numeric(df[c].str.replace(',', '.', regex=False), errors='coerce')
    for c in COLS_NUM:
        if c in df: df[c] = pd.to_numeric(df[c], errors='coerce')
    df['IR_29301_COD_GRD'] = df['IR_29301_COD_GRD'].str.lstrip('0')   # '044163' (2019-23) vs '41203' (2025)

    # --- 6. ANIO ---
    df.insert(0, 'ANIO', anio)

    # --- 7. SEXO ---
    df['SEXO'] = df['SEXO'].str.upper()
    df.loc[~df['SEXO'].isin(['HOMBRE', 'MUJER']), 'SEXO'] = 'DESCONOCIDO'

    # --- 7b. FALLECIDO (binario) a partir del tipo de alta ---
    if 'TIPOALTA' in df:
        df['FALLECIDO'] = df['TIPOALTA'].str.upper().str.contains('FALLEC', na=False).astype(int)
    else:
        df['FALLECIDO'] = pd.NA
    rep['FALLECIDO_sin_dato'] = int(df['FALLECIDO'].isna().sum())

     # --- 8. Diagnóstico principal: código -> palabras ---
    df['DIAGNOSTICO1'] = df['DIAGNOSTICO1'].str.upper()
    df['DIAGNOSTICO1_DESC'] = df['DIAGNOSTICO1'].map(DESC_CIE10)
    rep['DIAGNOSTICO1_sin_descripcion'] = int(df['DIAGNOSTICO1_DESC'].isna().sum())

    # Procedimientos CIE-9: ya no se usan -> se descartan las 30 columnas
    df = df.drop(columns=[f'PROCEDIMIENTO{i}' for i in range(1, 31)], errors='ignore')

    # --- 9. Posible duplicado (se marca y se elimina) ---
    clave = ['ID_BENEFICIARIO', 'COD_HOSPITAL', 'FECHA_INGRESO', 'DIAGNOSTICO1']
    df['POSIBLE_DUPLICADO'] = df.duplicated(clave, keep=False) & df['ID_BENEFICIARIO'].notna()
    rep['posibles_duplicados_marcados'] = int(df['POSIBLE_DUPLICADO'].sum())
    df = df.loc[~df['POSIBLE_DUPLICADO']].reset_index(drop=True)
    rep['posibles_duplicados_eliminados'] = rep['posibles_duplicados_marcados']

    # --- 10. Fecha de nacimiento: ya no se calcula edad ni tramo; se descarta ---
    df = df.drop(columns=['FECHA_NACIMIENTO', 'PROVINCIA'], errors='ignore')

    # --- 11. Hospital, comuna y región---
    df['COD_HOSPITAL'] = df['COD_HOSPITAL'].str.strip()
    df['HOSPITAL'] = df['COD_HOSPITAL'].map(NOMBRE_HOSPITAL)
    rep['hospital_sin_maestra'] = int(df['HOSPITAL'].isna().sum())
    rep['hospital_sin_maestra_cods'] = ', '.join(df.loc[df['HOSPITAL'].isna(), 'COD_HOSPITAL'].dropna().unique())
    df['HOSPITAL'] = df['HOSPITAL'].fillna('SIN CATALOGAR (' + df['COD_HOSPITAL'] + ')')

        # COMUNA = comuna de RESIDENCIA del paciente (columna original del GRD).
    df['COMUNA'] = df['COMUNA'].str.upper()
    rep['COMUNA_nula'] = int(df['COMUNA'].isna().sum())

    # REGION desde la COMUNA: cada comuna pertenece a una sola región (necesario para el mapa por comunas).
    df['REGION'] = df['COMUNA'].map(REGION_POR_COMUNA)
    rep['REGION_nula'] = int(df['REGION'].isna().sum())

    # --- 12. Orden: variables clave primero ---
    clave_primero = ['ANIO', 'COD_HOSPITAL', 'HOSPITAL', 'COMUNA', 'REGION', 'ID_BENEFICIARIO', 'SEXO', 'FALLECIDO',
                     'FECHA_INGRESO', 'MES_INGRESO', 'FECHAALTA',
                     'DIAGNOSTICO1', 'DIAGNOSTICO1_DESC', 'POSIBLE_DUPLICADO']
    resto = [c for c in df.columns if c not in clave_primero]
    df = df[clave_primero + resto]

    rep['filas_finales'] = len(df)
    return df, rep

## 4. Ejecutar todos los años y exportar

Se procesa un año a la vez (y se libera memoria) porque los archivos son grandes. Con `FECHAS_COMO_TEXTO = True` las fechas se guardan como texto `dd/mm/yyyy`.

In [18]:
reportes = []
grd = {}   # solo las columnas de interés, en memoria (nada se escribe a disco)

COLS_VER = ['ANIO', 'COD_HOSPITAL', 'HOSPITAL', 'COMUNA', 'REGION', 'SEXO', 'FALLECIDO',
            'FECHA_INGRESO', 'MES_INGRESO', 'FECHAALTA', 'DIAGNOSTICO1', 'DIAGNOSTICO1_DESC']

for anio in ANIOS:
    df, rep = limpiar_grd(anio)

    if FECHAS_COMO_TEXTO:
        df['FECHA_INGRESO'] = df['FECHA_INGRESO'].dt.strftime('%d/%m/%Y')

    reportes.append(rep)
    grd[anio] = df[COLS_VER].copy()
    print(f'{anio}: {rep["filas_originales"]:>7,} -> {rep["filas_finales"]:>7,} filas | '
          f'duplicados exactos eliminados: {rep["duplicados_exactos_eliminados"]}')
    del df

reporte = pd.DataFrame(reportes).set_index('anio').T
reporte

# Cantidad de valores nulos por columna en el GRD final de cada año.
nulos_por_anio = pd.DataFrame({anio: datos.isna().sum() for anio, datos in grd.items()})
nulos_por_anio

comunas_unicas = sorted(set().union(*[set(d['COMUNA'].dropna().unique()) for d in grd.values()]))
for c in comunas_unicas:
    print(repr(c))

2019:  96,439 ->  96,379 filas | duplicados exactos eliminados: 0
2020:  34,962 ->  34,800 filas | duplicados exactos eliminados: 128
2021:  34,618 ->  34,612 filas | duplicados exactos eliminados: 0
2022:  66,636 ->  66,622 filas | duplicados exactos eliminados: 2
2023:  88,782 ->  88,766 filas | duplicados exactos eliminados: 0
2024:  88,764 ->  88,757 filas | duplicados exactos eliminados: 0
'AISÉN'
'ALGARROBO'
'ALHUÉ'
'ALTO BIOBÍO'
'ALTO DEL CARMEN'
'ALTO HOSPICIO'
'ANCUD'
'ANDACOLLO'
'ANGOL'
'ANTOFAGASTA'
'ANTUCO'
'ANTÁRTICA'
'ARAUCO'
'ARICA'
'BUIN'
'BULNES'
'CABILDO'
'CABO DE HORNOS'
'CABRERO'
'CALAMA'
'CALBUCO'
'CALDERA'
'CALERA'
'CALERA DE TANGO'
'CALLE LARGA'
'CAMIÑA'
'CANELA'
'CARAHUE'
'CARTAGENA'
'CASABLANCA'
'CASTRO'
'CATEMU'
'CAUQUENES'
'CAÑETE'
'CERRILLOS'
'CERRO NAVIA'
'CHAITÉN'
'CHANCO'
'CHAÑARAL'
'CHEPICA'
'CHIGUAYANTE'
'CHILE CHICO'
'CHILLAN'
'CHILLAN VIEJO'
'CHIMBARONGO'
'CHOLCHOL'
'CHONCHI'
'CISNES'
'COBQUECURA'
'COCHAMÓ'
'COCHRANE'
'CODEGUA'
'COELEMU'
'COIHAIQUE'
'

## 5. Distribuciones de variables categóricas

Se muestran todos los valores distintos y su cantidad por año para `SEXO`, `COMUNA` y `MES_INGRESO`.

In [19]:
for columna in ['SEXO', 'COMUNA', 'MES_INGRESO']:
    print(f'\n{columna}')
    print(
        pd.concat(
            [datos[columna].value_counts(dropna=False).rename(anio) for anio, datos in grd.items()],
            axis=1,
        ).fillna(0).astype('int64').sort_index()
    )



SEXO
              2019   2020   2021   2022   2023   2024
SEXO                                                 
DESCONOCIDO     18      4      1      9     10      6
HOMBRE       50180  18979  18999  35481  46730  46041
MUJER        46181  15817  15612  31132  42026  42710

COMUNA
                 2019  2020  2021  2022  2023  2024
COMUNA                                             
AISÉN              76    44    56    73   105    61
ALGARROBO          56    27    32    38    64    76
ALHUÉ              28    20    12    24    30    33
ALTO BIOBÍO        19     3     4     9    26    22
ALTO DEL CARMEN    28    15    14    20    18    22
...               ...   ...   ...   ...   ...   ...
YUMBEL             69    35    19    51   103    46
YUNGAY             54    11    20    21    38    58
ZAPALLAR           10     5     3     7    15    14
ÑIQUÉN             83    35    44    95   100   114
ÑUÑOA             682   238   271   398   611   542

[345 rows x 6 columns]

MES_INGRESO
   

## 6. Verificación rápida

Se comprueban invariantes del GRD final: fecha de ingreso válida, mes consistente, ausencia de variables auxiliares y tramos válidos.

In [20]:
chk = pd.concat(grd.values(), ignore_index=True)
fecha_valida = chk['FECHA_INGRESO'].notna().all()
mes_consistente = chk['MES_INGRESO'].equals(chk['FECHA_INGRESO'].dt.month.astype('Int64'))
sin_auxiliares = not {'EDAD', 'FECHA_NACIMIENTO', 'TRAMO_EDAD'}.intersection(chk.columns)
sin_procedimientos = not any(c.startswith('PROCEDIMIENTO') for c in chk.columns)
anios_validos = set(chk['ANIO'].unique()) == set(ANIOS)

print(f'Fecha de ingreso sin nulos: {fecha_valida}')
print(f'MES_INGRESO coincide con FECHA_INGRESO: {mes_consistente}')
print(f'Sin EDAD, FECHA_NACIMIENTO ni TRAMO_EDAD: {sin_auxiliares}')
print(f'Sin columnas PROCEDIMIENTO*: {sin_procedimientos}')
print(f'Solo años 2019-2024: {anios_validos}')

region_ok = chk['REGION'].isna().mean() < 0.01
sin_region = chk.loc[chk['REGION'].isna() & (chk['COMUNA'].fillna('') != 'DESCONOCIDO'), 'COMUNA'].unique()
print(f'REGION con menos de 1% de nulos: {region_ok}')
print(f'Comunas sin región en el catálogo (salvo DESCONOCIDO): {len(sin_region)}')

assert region_ok
assert len(sin_region) == 0, sin_region

assert fecha_valida
assert mes_consistente
assert sin_auxiliares
assert sin_procedimientos
assert anios_validos
print(f'\nValidacion OK: {len(chk):,} filas y {chk.shape[1]} columnas')

Fecha de ingreso sin nulos: True
MES_INGRESO coincide con FECHA_INGRESO: True
Sin EDAD, FECHA_NACIMIENTO ni TRAMO_EDAD: True
Sin columnas PROCEDIMIENTO*: True
Solo años 2019-2024: True
REGION con menos de 1% de nulos: True
Comunas sin región en el catálogo (salvo DESCONOCIDO): 0

Validacion OK: 409,936 filas y 12 columnas
